# 4. モデルを作って予測する

前処理で作った、全ての値が数値で欠損のないデータを使う。
乗客の属性から生存を予測するモデルを学習し、評価用データの446人について
生き残る確率を出す。

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

# 03 で保存した加工後データ。無ければ 03 を実行して作る
train = pd.read_csv("../data/processed/train_processed.csv", index_col=0)
test = pd.read_csv("../data/processed/test_processed.csv", index_col=0)

print("train:", train.shape, " test:", test.shape)

## 1. 答えと手がかりに分ける

前処理を終えた `train` は 11 列ある。このうち `survived` だけが**答え**で、
残りの 10 列が、その答えを導くための**手がかり**になる。

| id | survived | pclass | age | sex_female | ... |
| --- | --- | --- | --- | --- | --- |
| 3 | 1 | 1 | 35.0 | True | ... |
| 4 | 0 | 3 | 35.0 | False | ... |

学習させるときは、この2つを**別々に渡す**。

```
        train（11 列）
  ┌──────────┬──────────────────────────┐
  │ survived │ pclass  age  sex_female  │
  │          │ sibsp   parch  fare  ... │
  └──────────┴──────────────────────────┘
       │                   │
       ▼                   ▼
       y                   X
    答え（1 列）      手がかり（10 列）
```

問題集にたとえると、`X` が問題文で、`y` が巻末の解答にあたる。
両方を突き合わせて見せることで、「3等客室の男性はたいてい 0」「女性はたいてい 1」
といった対応をモデルに見つけさせる。

答えを混ぜたまま渡してはいけない理由も、このたとえで分かる。
問題文の中に解答が書いてあったら、考えずにそれを読むだけになってしまう。

### なぜ `X` と `y` なのか

変数名は何でもよいが、機械学習のコードではこの2文字がほぼ固定で使われる。

- `X` が**大文字**なのは、複数の列を持つ表だから（数学で行列を大文字で書く慣習）
- `y` が**小文字**なのは、1列しかないから

他人のコードを読むときも同じ名前で出てくるので、そういうものとして覚えてしまってよい。

### `drop` で列を落とす

`drop` は行にも列にも使えるので、**どちらを落とすのかを伝える**必要がある。

```python
train.drop(columns=["survived"])   # 列を落とす
train.drop(index=[3, 4])           # 行を落とす
```

`axis=1`（列）、`axis=0`（行）という番号で指定する書き方もあり、他人のコードでは
こちらをよく見かける。番号は `shape` が返す `(行数, 列数)` の並び順に対応している。

結果は同じなので、読んで意味が分かる `columns=` の方をここでは使う。

In [ ]:
y = train["survived"]  # 目的変数
X = train.drop(columns=["survived"])  # 説明変数（survived を除いた残り全部）

print("X:", X.shape, " y:", y.shape)
print("X の列:", list(X.columns))

# test には元から survived が無いので、X と同じ 10 列になっている
print("test と列の並びが一致:", list(X.columns) == list(test.columns))

# 並び順の一致を確認しているのは、モデルが列名を見ていないため。
# 「左から 2 番目は年齢」と位置で覚えるので、test だけ順序が違うと
# 年齢のつもりで運賃を読んでしまい、気づかないまま間違った予測になる

## 2. 採点用のデータを取り分ける

`train` の445人を全部学習に使ってしまうと、**モデルの出来を測る手段が無くなる**。

学習に使ったデータで精度を測っても、答えを覚えただけのモデルが満点を取れてしまい、
知らない乗客に通用するかが分からない。

そこで445人を2つに分ける。

- **学習用**（8割）— モデルに見せる
- **検証用**（2割）— モデルには見せず、採点にだけ使う

### `train` という言葉が2階層で出てくる

紛らわしいので、全体の関係を整理しておく。

```
891 人（元の名簿）
├── train.csv  445 人  ← 答え（survived）がある
│   ├── X_train  356 人  ← モデルに見せる
│   └── X_valid   89 人  ← 隠しておいて採点に使う
└── test.csv   446 人  ← 答えが無い。SIGNATE が持っている
```

`train`（445人）を自分でさらに 4:1 に割ったのが `X_train` と `X_valid`。

`valid` は **validation**（検証）の略。なぜ `test` と呼ばないかというと、
その名前が既に埋まっているため。本来この役割は3つに分かれる。

| 区分 | 役割 | 今回 |
| --- | --- | --- |
| train | 見せて学ばせる | `X_train` |
| validation | 手元で成績を測り、改良の効果を判定する | `X_valid` |
| test | 最後の実力測定 | `test.csv`（答えは手元に無い） |

`X_valid` の点数は何度測ってもよい。提出前の品質チェックに使う手元の物差し。

In [ ]:
# train_test_split はデータをランダムに2つへ分ける
#   test_size=0.2  : 2割を検証用に取り分ける
#   random_state=0 : 乱数の種を固定する。付けないと実行のたびに分け方が変わり、
#                    精度が上下したとき改良の成果か偶然かを区別できなくなる
#   stratify=y     : 分けた後も生存者の割合が元と同じになるようにする。
#                    偏ると採点の条件が変わってしまうため
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

print("学習用:", X_train.shape, " 検証用:", X_valid.shape)

## 3. モデルを作る

**ロジスティック回帰**を使う。それぞれの列に重みを付けて足し合わせ、
その合計を 0〜1 の確率に変換する仕組み。「女性なら生存側に大きく傾ける」といった
判断を、データから重みとして学び取る。

ただし、そのまま渡すと問題が起きる。列ごとに値の桁が違いすぎるためだ。

| 列 | 値の範囲 |
| --- | --- |
| `fare` | 0 〜 512 |
| `age` | 0.67 〜 80 |
| `sex_female` | 0 か 1 |

この状態では運賃の数値だけが極端に大きく、計算が安定しない。
実際、何も手を打たないと「計算が終わらなかった」という警告が出る。

そこで **標準化** を挟む。各列を「平均0・ばらつき1」に揃える変換で、
桁の違いをなくして同じ土俵に乗せる。

### 標準化すると値がどう変わるか

変換後の数字は「**平均から標準偏差いくつ分ずれているか**」を表す。
歳や運賃といった単位が消えて、どの列も同じ物差しに乗る。

なお 0〜1 に収める変換ではない。**マイナスにも 1 超えにもなる。**

In [ ]:
# パイプラインの中で起きているのと同じ変換を、目で見るために単体で実行する
#   .fit() で平均と標準偏差を覚え、.transform() でその基準に沿って変換する
scaler = StandardScaler().fit(X_train)
after = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)

cols = ["age", "fare", "sex_female"]
print("変換前");  print(X_train[cols].head(3))
print()
print("変換後");  print(after[cols].head(3).round(3))

# 出力の見方（1 人目の id 393）
#   age  23.0    → -0.452  平均 28.89 歳より少し若いので、わずかにマイナス
#   fare 113.275 → +1.406  平均 35.31 より大幅に高いので、大きくプラス
#   もとの桁は 23 と 113 で 5 倍違ったが、変換後はどちらも -1 〜 1 付近に収まる

In [ ]:
# 列ごとに、変換前後の平均とばらつきを比べる
pd.DataFrame({
    "変換前 平均": X_train[cols].mean(),
    "変換前 std": X_train[cols].std(ddof=0),
    "変換後 平均": after[cols].mean(),
    "変換後 std": after[cols].std(ddof=0),
}).round(3)

# 出力の見方
#   変換前は平均が 28.9 / 35.3 / 0.4 とばらばらで、ばらつきも 13 / 55 / 0.5 と桁が違う
#   変換後はどの列も平均 0・ばらつき 1 に揃う。これが「同じ土俵に乗せる」ということ

In [ ]:
# 変換後にどこまで値が動くか
after[cols].agg(["min", "max"]).round(2)

# 出力の見方
#   age  -2.17 〜 3.92、fare -0.64 〜 8.60
#   0 〜 1 には収まらない。fare の 8.60 は、512 を払った人が
#   「平均より標準偏差 8.6 個分も高い」という飛び抜けた存在であることを示している

In [ ]:
# make_pipeline は複数の処理を1本に繋げる
#   ここでは 標準化 → ロジスティック回帰 の順に通る
#
# 繋げる理由は、標準化の基準を学習用データだけから決めるため。
# 別々に書くと、検証用や評価用のデータまで含めて平均を計算してしまいがちで、
# 本来知らないはずの情報が混ざり込む
model = make_pipeline(StandardScaler(), LogisticRegression())

# .fit(手がかり, 答え) で学習する。これだけでモデルができる
#   セルの最後に置くと、組み立てた工程がそのまま表示される
model.fit(X_train, y_train)

### `.fit()` は何をしたのか

モデルの正体は、**列ごとの重み（係数）と切片**という数字の並びでしかない。
`.fit()` はこれを決める作業をしている。

始める前は全部 0 で、誰を入れても答えが 0.5 になる状態。そこから、

1. 今の係数で356人の生存確率を計算する
2. 実際の `survived` と比べて、ズレ（誤差）を測る
3. 誤差が減る方向に係数を少し動かす
4. 1 に戻る

を繰り返し、誤差がこれ以上減らなくなったら終わる。
356人のデータそのものはモデルに残らない。**残るのは絞り出された数字だけ。**

In [ ]:
# fit が決めた数字を見る。model[-1] はパイプラインの最後、ロジスティック回帰本体
pd.Series(model[-1].coef_[0], index=X.columns).round(3)

# 出力の見方
#   プラスなら生存側、マイナスなら死亡側に働く重み
#   sex_female +0.654 / sex_male -0.654、pclass -0.772 が大きい
#   これに切片（下のセル）を足したものが、判定に使われる式のすべて

In [ ]:
print("切片:", round(model[-1].intercept_[0], 3))

# 切片は、どの列にも紐付かない下駄。
# 全員に一律で足される値で、全体としてどちら寄りに判定するかを調整している

## 4. 成績を確かめる

`.score()` は正解率を返す。**見せていない検証用データでの成績**が本当の実力になる。

比較の基準として、02 で見た「全員が助からなかったと答えるだけのモデル」の
正解率も並べる。これを超えていなければ、何も学習できていないのと同じ。

In [ ]:
print("検証用データでの正解率:", round(model.score(X_valid, y_valid), 4))
print("学習用データでの正解率:", round(model.score(X_train, y_train), 4))
print("全員「助からない」と答えた場合:", round(1 - y_valid.mean(), 4))

# 出力の見方
#   検証 0.7528 に対して基準 0.5955。約 16 ポイント上回っており、法則を捉えられている
#   学習側が高いのは、係数がその356人に合わせて決められたため。
#   自分に合わせて作った物差しで自分を測っているようなもの

### 学習用と検証用の違い

どちらも**同じ係数を使った同じ計算**で、違うのは誰に当てはめたかだけ。

| | 人数 | モデルは見たか |
| --- | --- | --- |
| 学習用 | 356 | 見た。答えも一緒に |
| 検証用 | 89 | **見ていない** |

問題集にたとえると、学習用は答え合わせ済みの例題、検証用は初見の模試。
**実力を表すのは検証用の方**で、提出したときの点数もこれが目安になる。

ただし**1回の分割の数字を細かく読みすぎない。** 検証用は89人しかいないので、

```
1 人の正誤 = 1.12 ポイント
```

数ポイントの差は数人分でしかない。実際、分け方を変えると検証精度は
0.72〜0.89 の範囲で動き、検証の方が高く出る回もある。

学習用と検証用の差が**大きく開いたとき**だけ、暗記に寄ったサインとして読む。
例えば決定木を制限なしで使うと学習 0.99 / 検証 0.72 まで開く。

### 答えを知っているのに、なぜ学習用でも 100% にならないのか

**予測するとき、モデルに答えは渡っていないから。**

```
学習時: 10 列 + 答え  →  係数を決める
予測時: 10 列だけ     →  係数に通して確率を出す
```

さらに、モデルが持てるのは係数と切片という数字の並びだけで、
356人分の答えを保存する場所がない。できるのは
「こういう属性の人はだいたいこう」という**法則の形に圧縮すること**だけ。

だから例外は必ず外す。

- 3等の男性は 175 人中 151 人（86.3%）が亡くなっている。法則としては「死亡」に
  賭けるのが正しく、それでも生還した 24 人は外れる
- 1等の女性は 53 人中 94.3% が生還している。ほぼ確実に助かる属性なので、
  亡くなった人は外れる

もうひとつの理由は、ロジスティック回帰が**単純**なこと。
できるのは各列に重みを掛けて足すだけで、「3等でも女性で子連れなら助かりやすい」
といった**条件の組み合わせ**は表現できない。

この単純さは弱点であると同時に、覚え込みすぎない強さでもある。

## 5. 全データで学習し直して予測する

成績の見当が付いたので、**445人全員**を使って学習し直す。

検証用に取り分けていた2割も、本番のモデルでは使わない手はない。
学習に使うデータは多いほどよく、成績の確認はもう済んでいる。

In [ ]:
# 同じ構成のモデルを、今度は train 全体で学習する
model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X, y)

In [ ]:
# .predict_proba() は確率を返す。列が2つあることに注意
proba = model.predict_proba(test)

print("形:", proba.shape)
print("先頭1人の値:", proba[0].round(4), " 合計:", proba[0].sum())

# 出力の見方
#   左の列 = 助からない確率、右の列 = 生き残る確率。2つ足すと必ず 1 になる
#   提出に必要なのは生存確率なので、右の列だけを取り出す

In [ ]:
# [:, 1] は「全ての行の、1番目の列」という指定（0 から数えるので 1 が右の列）
pred = model.predict_proba(test)[:, 1]

print("先頭5人の生存確率:", pred[:5].round(6))
print("生存確率が 0.5 を超えた人数:", (pred > 0.5).sum(), "/", len(pred))

## 6. どの列が効いたか

学習した重み（係数）を見ると、モデルが何を根拠に判断しているかが分かる。
標準化してあるので、列どうしで大きさを比べられる。

プラスなら生存側、マイナスなら死亡側に働く。

In [ ]:
# model[-1] はパイプラインの最後、ロジスティック回帰本体
coef = pd.Series(model[-1].coef_[0], index=X.columns).sort_values()
coef

# 出力の見方
#   pclass -0.797  : 等級の数字が大きい（下の客室）ほど死亡側。最も強い
#   sex_male -0.637 / sex_female +0.637 : 男性は死亡側、女性は生存側
#   age -0.445     : 高齢ほど死亡側
#   02 の相関係数では弱く見えた age が、ここでは3番目に効いている。
#   相関係数が1列ずつ単独で見るのに対し、モデルは他の列の影響を差し引いた上で
#   age の寄与を測るため

### 1人分を手で追ってみる

`predict_proba` の中で起きていることを、実際に分解する。
やっているのは「標準化した値 × 係数」を全部足して、確率に変換するだけ。

In [ ]:
import numpy as np

# 評価用データの1人目を取り出す
person = test.iloc[[0]]
scaler, clf = model[0], model[1]

z = scaler.transform(person)[0]  # 標準化した値
contrib = pd.Series(z * clf.coef_[0], index=X.columns)  # 値 × 係数

print("寄与が大きい順:")
print(contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(5).round(3))
print()
total = contrib.sum() + clf.intercept_[0]
print("合計 + 切片 =", round(total, 3))
print("確率に変換  =", round(1 / (1 + np.exp(-total)), 6))
print("predict_proba と一致:", np.isclose(1/(1+np.exp(-total)), model.predict_proba(person)[0, 1]))

# 出力の見方
#   マイナスの寄与が積み上がると合計が下がり、確率も下がる
#   最後の変換で、どんな合計値でも必ず 0〜1 に収まる
#   これを 446 人分繰り返したものが pred

## 7. 予測結果を保存する

05 で提出用のファイルに整えるため、id と紐付けて書き出す。

書き出す `pred_test.csv` は、**446人分の生存確率**を id と並べただけのファイル。

```
id,pred
0,0.101915     ← id 0 の人は 10.2% の確率で生還
1,0.931212
```

まだ提出できる形ではない。提出用はヘッダ行を付けない決まりになっている。
確率の値はそのまま使うので、整形は 05 で行う。

`id` を一緒に書き出しているのが要点で、並び順だけに頼ると、
どこかで行が入れ替わったときに**別人の予測を提出してしまう**。

In [ ]:
# test.index が乗客の id。並び順に依存せず対応が取れるよう、id を付けて保存する
pd.Series(pred, index=test.index, name="pred").to_csv(
    "../data/processed/pred_test.csv", index=True
)

print("保存した")

## この章のまとめ

- `survived` を目的変数 `y`、残り10列を説明変数 `X` として分けた
- 445人のうち2割を採点用に取り分けて学習し、**検証用データで 75.3%** の正解率。
  基準となる 59.6%（全員「助からない」と答えた場合）を約16ポイント上回った
- 列によって値の桁が違いすぎるため、標準化を挟んでいる。
  入れないと計算が収束せず警告が出る
- 成績を確認した後、445人全員で学習し直してから予測した
- 効いている順は客室クラス、性別、年齢。
  単独では弱く見えた年齢が、他の列の影響を除くと3番目に効いている
- 446人の生存確率を `data/processed/pred_test.csv` に保存した